# CMS Patient Matching — quick demo (v1)

This notebook shows the **new CMS matching algorithm** running end-to-end on a
handful of patients, with **no database or external infrastructure** required.

## What's different from the old approach
The previous person-matching engine produced a single **weighted score** and
compared it to a `0.955` threshold — one wrong or missing field could drag an
otherwise-good record below the bar.

The CMS approach (Proposal v3.2.2, Table 2) is fundamentally different:

- Matching is **combination-based**: a query matches if it satisfies **any** of the
  approved field combinations (e.g. *First + Last + DOB + Street*), each of which is
  pre-validated to a collision probability at or below the framework threshold.
- A match is only returned when it resolves to **exactly one** candidate
  (**uniqueness check**) — otherwise it is flagged **ambiguous** and withheld.
- Constrained **fuzzy** matching (Damerau-Levenshtein ≤ 1, names/street only, ≥5 chars).
- A **generational-suffix conflict vetoes** a match (Jr vs Sr).
- Disagreement on a field that is **not part of the matched combination does not
  block** the match.

This demo uses Imran's `MatchingManager`/`MatchingEngine` (the real engine) plus the
new `InMemoryBackend`, so you can experiment without standing up DuckDB/Mongo/etc.


In [ ]:
from typing import Any, Dict, List

from patient_matching.matching import (
    InMemoryBackend,
    MatchingManager,
    MatchOutcome,
)

## 1. Build a small patient "database"

These dicts are **already normalized** (lowercase, punctuation stripped, phones in
E.164) — exactly what `FieldExtractor` expects. In production, raw FHIR Patients are
run through `NormalizationManager` first (case-folding, diacritics, placeholder
suppression, Project US@ addresses, etc.); we skip that here to keep the demo
dependency-free and focused on the matching logic.

In [ ]:
def patient(pid, given, family, dob, street=None, *, phone=None, email=None, suffix=None):
    """Build a minimal, already-normalized FHIR Patient dict."""
    p: Dict[str, Any] = {
        "resourceType": "Patient",
        "id": pid,
        "name": [{"given": [given]}],
        "birthDate": dob,
    }
    if family:
        p["name"][0]["family"] = family
    if suffix:
        p["name"][0]["suffix"] = [suffix]
    if street:
        p["address"] = [{"line": [street]}]
    telecom = []
    if phone:
        telecom.append({"system": "phone", "value": phone})
    if email:
        telecom.append({"system": "email", "value": email})
    if telecom:
        p["telecom"] = telecom
    return p


DATABASE: List[Dict[str, Any]] = [
    patient("patient-1", "john", "smith", "1985-03-15", "123 main st",
            phone="+16175551234", email="john.smith@example.com"),
    patient("patient-2", "maria", "garcia", "1990-07-22", "500 oak ave", phone="+16175559999"),
    patient("patient-3", "robert", "jones", "1972-11-05", "77 elm rd"),
    # Two DIFFERENT people who share First + DOB + Phone (used for the ambiguity demo):
    patient("patient-4", "alex", "lee", "2000-01-01", "10 pine st", phone="+16175550000"),
    patient("patient-5", "alex", "park", "2000-01-01", "22 birch st", phone="+16175550000"),
    # Has a generational suffix (used for the suffix-veto demo):
    patient("patient-7", "william", "brownlee", "1960-06-06", "12 king st", suffix="sr"),
]

backend = InMemoryBackend(DATABASE)
manager = MatchingManager(backend=backend)
print(f"Loaded {len(backend)} patients; evaluating {manager.rule_count} Table 2 rules.")

In [ ]:
def show(title, query):
    """Run a match and print the result in a readable form."""
    r = manager.match(query)
    ids = [p["id"] for p in r.matched_patients]
    print(title)
    print(f"  outcome={r.outcome.value}  rule={r.matched_rule_id}  type={r.match_type}  "
          f"unique={r.is_unique}  candidates={r.candidate_count}  matched={ids}")
    return r

## 2. Scenario walk-through

### 2.1 Exact, unique match
Query agrees with one record on **First + Last + DOB + Street** (Table 2 rule 01).

In [ ]:
_ = show("Exact unique match", patient("q1", "john", "smith", "1985-03-15", "123 main st"))

### 2.2 Fuzzy match
The last name is mistyped `smyth` (Damerau-Levenshtein distance 1 from `smith`,
≥ 5 characters). Rule 01 allows fuzzy on the name fields, so it still matches —
and the result is tagged `type=fuzzy` for the audit trail.

In [ ]:
_ = show("Fuzzy last-name match", patient("q2", "john", "smyth", "1985-03-15", "123 main st"))

### 2.3 Ambiguous — withheld by the uniqueness check
Two different people share **First + DOB + Phone** (rule 11). Because the query does
not resolve to a single person, the algorithm returns **AMBIGUOUS and releases
nothing** — the core safeguard against wrong-patient matches.

In [ ]:
_ = show("Ambiguous (not released)", patient("q3", "alex", None, "2000-01-01", phone="+16175550000"))

### 2.4 No match
Nobody in the database satisfies any approved combination.

In [ ]:
_ = show("No match", patient("q4", "nancy", "nobody", "1999-09-09", "999 nowhere ln"))

### 2.5 A disagreeing NON-qualifying field does not block the match
The query matches Maria on **First + Last + DOB + Street** (rule 01). It also carries
a **different phone number** than the record — but phone is not part of rule 01, so it
does not negate the match. (Under the old score-and-threshold approach, a mismatched
field like this could pull the score below the bar.)

In [ ]:
_ = show("Extra field disagrees, still matches",
         patient("q5", "maria", "garcia", "1990-07-22", "500 oak ave", phone="+19998887777"))

### 2.6 Suffix conflict vetoes a match
The query would match William on rule 01, but the query is **Jr** and the record is
**Sr**. A generational-suffix conflict is a deterministic veto (proposal §B.5).

In [ ]:
_ = show("Suffix conflict veto",
         patient("q6", "william", "brownlee", "1960-06-06", "12 king st", suffix="jr"))

## 3. Audit trail (proposal §VII)
Every evaluation records the rule, match type (exact/fuzzy), and per-field outcomes,
so any decision can be reconstructed for audit.

In [ ]:
result = manager.match(patient("q1", "john", "smith", "1985-03-15", "123 main st"))
for ev in result.rule_evaluations:
    if ev.matched:
        print(f"rule {ev.rule_id}: type={ev.match_type} fields={ev.field_outcomes}")

## Notes & next steps
- This engine implements **CMS Proposal v3.2.2** (26 Table 2 combinations). The current
  draft is **v3.3.0** (37 combinations, ±1-day DOB tolerance, insurance Member/Subscriber
  ID rules with payer namespaces). Extending `table2_rules.py` to v3.3.0 is the natural
  follow-up.
- Gender/sex is intentionally **not** a matching field in the CMS approach.
- To run the automated tests for this module:
  ```
  pytest patient_matching/matching/tests
  ```
- Swap `InMemoryBackend` for a production backend (DuckDB/Mongo/Elasticsearch) to run at
  scale — the `MatchingManager` API is identical.